In [1]:
import json

with open("/Users/andrewweiland/UCCS_REU/quantum-optimizer-orchestration/brute_force_chains/cheap_optimizers/initial_test/optimize_only/reports/results_20260609_155120.json") as f:
    data = json.load(f)

print(type(data))

<class 'dict'>


In [27]:
print(data.keys())

dict_keys(['metadata', 'results', 'failures'])


In [32]:
import pandas as pd
df = pd.DataFrame(data["results"])

print(df.iloc[10]["metadata"])
df.iloc[0]

{'chain_name': 'fast_chain_sample_tket__qiskit_standard', 'num_steps': 2, 'steps': [{'name': 'tket', 'runner_type': 'tket'}, {'name': 'qiskit_standard', 'runner_type': 'qiskit_standard'}], 'step_durations': [0.9253990000579506, 0.01386212499346584], 'step_improvements': [-100.0, -91.07142857142857], 'total_duration_seconds': 0.9419590829638764}


circuit                                                      qft_8
runner                                                   qiskit_ai
optimizer                                                    chain
label                                              chain_qiskit_ai
metrics          {'depth': 31, 'two_qubit_gates': 52, 'two_qubi...
metadata         {'chain_name': 'fast_chain_sample_qiskit_ai', ...
artifact_path    brute_force_chains/cheap_optimizers/initial_te...
Name: 0, dtype: object

In [33]:
df["opt_level"] = df["metadata"].apply(lambda x: x.get("num_steps"))
df["opt_1"] = df["metadata"].apply(
    lambda x: x["steps"][0]["name"] if len(x["steps"]) > 0 else pd.NA
)
df["opt_2"] = df["metadata"].apply(
    lambda x: x["steps"][1]["name"] if len(x["steps"]) > 1 else pd.NA
)
df["opt_3"] = df["metadata"].apply(
    lambda x: x["steps"][2]["name"] if len(x["steps"]) > 2 else pd.NA
)
df["opt_chain"] = df.apply(
    lambda row: "__".join(
        [str(x) for x in [
            row["circuit"],
            row["opt_1"],
            row["opt_2"],
            row["opt_3"]
        ] if pd.notna(x)]
    ),
    axis=1
)
df["runtime"] = df["metadata"].apply(lambda x: x.get("total_duration_seconds"))
df["final_depth"] = df["metrics"].apply(lambda x: x.get("depth"))
df["final_two_qubit_gate_count"] = df["metrics"].apply(lambda x: x.get("two_qubit_gates"))
df["final_two_qubit_depth"] = df["metrics"].apply(lambda x: x.get("two_qubit_depth"))
df["final_total_gates"] = df["metrics"].apply(lambda x: x.get("total_gates"))
df.drop(columns=["metadata", "metrics", "runner", "label", ], inplace=True)
df.head()

,circuit,optimizer,artifact_path,opt_level,opt_1,opt_2,opt_3,opt_chain,runtime,final_depth,final_two_qubit_gate_count,final_two_qubit_depth,final_total_gates
0,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,1,qiskit_ai,<NA>,<NA>,qft_8__qiskit_ai,1.002103,31,52,56,60
1,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,1,qiskit_standard,<NA>,<NA>,qft_8__qiskit_standard,0.020029,112,108,64,281
2,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,1,tket,<NA>,<NA>,qft_8__tket,1.145792,51,56,26,129
3,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,2,qiskit_ai,qiskit_ai,<NA>,qft_8__qiskit_ai__qiskit_ai,0.418219,31,51,51,59
4,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,2,qiskit_ai,qiskit_standard,<NA>,qft_8__qiskit_ai__qiskit_standard,0.178461,100,101,54,263


In [34]:
df.to_csv("test_results.csv", index=False)